<a href="https://colab.research.google.com/github/pengin-cmd/my-colab-notebooks/blob/main/%E3%82%B3%E3%83%B3%E3%83%9A2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [15]:
!pip install xgboost catboost optuna lightgbm

In [16]:
!pip install optuna

In [17]:
pip install lifetimes

KeyboardInterrupt: 

In [ ]:
import glob
import os
import re
import matplotlib.pyplot as plt
import pandas as pd

# 1. 実行中のディレクトリ（または特定のフォルダ）内から 'result_' で始まるCSVファイルを自動検索
# ※スクリプトと同じフォルダにあるファイルを対象にする場合はこれで動作します
csv_files = glob.glob("result_*.csv")

data = []
# ファイル名から「番号」「ACU」「Loss」を抽出する正規表現パターン
pattern = r"result_(\d+)\(ACU([\d\.]+),Loss([\d\.]+)\)\.csv"

# 2. 自動取得したファイル群から情報を抽出
for f in csv_files:
    # パス（フォルダ名など）が含まれている場合を考慮し、純粋なファイル名のみを抽出
    basename = os.path.basename(f)

    match = re.match(pattern, basename)
    if match:
        num = int(match.group(1))
        acu = float(match.group(2))
        loss = float(match.group(3))
        data.append({"result_num": num, "ACU": acu, "Loss": loss})

# データが1つも見つからなかった場合の処理
if not data:
    print(
        "条件（result_数字(ACU...,Loss...).csv）に一致するファイルが見つかりませんでした。"
    )
else:
    # 3. データを結果の番号順（1, 2, 3...）に並び替えてDataFrameに変換
    df_metrics = pd.DataFrame(data).sort_values(by="result_num").reset_index(drop=True)

    # 4. 散布図のプロット
    fig, ax = plt.subplots(figsize=(8, 6))

    ax.scatter(
        df_metrics["ACU"],
        df_metrics["Loss"],
        color="crimson",
        s=100,
        edgecolors="black",
        alpha=0.8,
        label="Results",
    )

    # 各プロット点に対応する結果番号を注釈として自動追加
    for i, txt in enumerate(df_metrics["result_num"]):
        ax.annotate(
            f"result_{txt}",
            (df_metrics["ACU"].iloc[i], df_metrics["Loss"].iloc[i]),
            textcoords="offset points",
            xytext=(0, 10),
            ha="center",
            fontsize=9,
            fontweight="bold",
        )

    # 軸ラベルとタイトルの設定
    ax.set_xlabel("ACU", fontsize=12)
    ax.set_ylabel("Loss", fontsize=12)
    ax.set_title(
        "Distribution of ACU vs Loss (Auto-loaded)", fontsize=14, fontweight="bold"
    )
    ax.grid(True, linestyle="--", alpha=0.6)

    # レイアウトの自動調整
    plt.tight_layout()

    # 5. 画像の保存
    output_image = "acu_loss_distribution_auto.png"
    plt.savefig(output_image, dpi=300)
    print(
        f"全 {len(df_metrics)} 個のファイルを自動認識し、散布図を '{output_image}' として保存しました。"
    )

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# データの読み込み
df = pd.read_csv('train.csv')

# 数値型の列のみを抽出（文字列や日付データを除外）
numeric_df = df.select_dtypes(include=['number'])

# 相関行列の計算
corr = numeric_df.corr()

# ヒートマップの作成
plt.figure(figsize=(12, 10))
sns.heatmap(corr, annot=False, cmap='coolwarm', fmt=".2f")
plt.title('Correlation Heatmap of train.csv')
plt.tight_layout()

# 画像として保存
plt.savefig('heatmap.png')

In [18]:
import pandas as pd
import numpy as np
import warnings
import optuna
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score, log_loss
from lightgbm import LGBMClassifier, early_stopping as lgb_early_stopping

warnings.filterwarnings('ignore')

# ==========================================
# 0. 絶対にリークしない事前処理（行単位の処理）
# ==========================================
def preprocess_safe_row_wise(df):
    df = df.copy()

    # --- 1. 日付・年齢に関する特徴量 ---
    base_date = pd.to_datetime('2015-01-01')
    if 'registration_date' in df.columns:
        df['days_since_registration'] = (base_date - pd.to_datetime(df['registration_date'])).dt.days
        df = df.drop('registration_date', axis=1)

    if 'age' in df.columns:
        df['prime_young_age_flag'] = ((df['age'] >= 25) & (df['age'] <= 39)).astype(int)
        df['older_peak_flag'] = ((df['age'] >= 55) & (df['age'] <= 60)).astype(int)

    # --- 2. 年収に関する特徴量 ---
    if 'annual_income' in df.columns:
        df['high_income_th'] = (df['annual_income'] >= 70000).astype(int)
        df['middle_income_cluster'] = ((df['annual_income'] >= 30000) & (df['annual_income'] <= 65000)).astype(int)

    # --- 3. 支出に関する特徴量 ---
    if 'spend_wines' in df.columns:
        df['wine_heavy_spender'] = (df['spend_wines'] >= 600).astype(int)
    if 'spend_meat' in df.columns:
        df['meat_heavy_spender'] = (df['spend_meat'] >= 200).astype(int)

    # --- 4. マーケティング理論・相関分析に基づく高度特徴量 ---
    spend_cols = ['spend_wines', 'spend_fruits', 'spend_meat', 'spend_fish', 'spend_sweets', 'spend_gold']
    if all(c in df.columns for c in ['store_purchases', 'web_purchases', 'catalog_purchases'] + spend_cols):
        total_purchases = df['store_purchases'] + df['web_purchases'] + df['catalog_purchases']
        total_spend = df[spend_cols].sum(axis=1)

        df['average_order_value'] = total_spend / (total_purchases + 1)

        for cat in spend_cols:
            df[f'{cat}_share'] = df[cat] / (total_spend + 1)

        df['hhi_category_concentration'] = sum(df[f'{cat}_share']**2 for cat in spend_cols)

    if all(c in df.columns for c in ['monthly_web_visits', 'deals_purchases']):
        df['web_discount_hunter_idx'] = df['monthly_web_visits'] * df['deals_purchases']

    return df

# ==========================================
# 0.5 カテゴリ変数の型変換
# ==========================================
def convert_categorical_types(df, cat_cols):
    df = df.copy()
    for col in cat_cols:
        if col in df.columns:
            df[col] = df[col].astype('category')
    return df

# ==========================================
# 1. Optuna による LightGBM パラメータチューニング
# ==========================================
def tune_lightgbm_with_optuna(X_raw, y, cat_cols, n_trials=10):
    print("\n--- [1. Optuna パラメータ最適化 (LightGBM)] ---")

    def objective(trial):
        params = {
            'random_state': 42,
            'n_estimators': 500,
            'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
            'max_depth': trial.suggest_int('max_depth', 3, 8),
            'num_leaves': trial.suggest_int('num_leaves', 15, 63),
            'min_child_samples': trial.suggest_int('min_child_samples', 10, 50),
            'class_weight': 'balanced',
            'verbose': -1
        }

        skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
        oof = np.zeros(len(X_raw))

        for tr_idx, va_idx in skf.split(X_raw, y):
            X_tr, y_tr = X_raw.iloc[tr_idx].copy(), y.iloc[tr_idx]
            X_va, y_va = X_raw.iloc[va_idx].copy(), y.iloc[va_idx]

            med = X_tr['annual_income'].median()
            X_tr['annual_income'] = X_tr['annual_income'].fillna(med)
            X_va['annual_income'] = X_va['annual_income'].fillna(med)

            X_tr = convert_categorical_types(X_tr, cat_cols)
            X_va = convert_categorical_types(X_va, cat_cols)

            model = LGBMClassifier(**params)
            model.fit(
                X_tr, y_tr,
                eval_set=[(X_va, y_va)],
                callbacks=[lgb_early_stopping(stopping_rounds=30, verbose=False)]
            )
            oof[va_idx] = model.predict_proba(X_va)[:, 1]

        return roc_auc_score(y, oof)

    optuna.logging.set_verbosity(optuna.logging.WARNING)
    study = optuna.create_study(direction='maximize')
    study.optimize(objective, n_trials=n_trials)

    best_params = study.best_params
    best_params.update({'n_estimators': 1000, 'class_weight': 'balanced', 'verbose': -1})
    print(f"✅ 最適パラメータ: {best_params}")
    return best_params

# ==========================================
# 2. クロスバリデーションでの学習関数 (シード別)
# ==========================================
def train_lgb_seed(X_raw, y, X_test_raw, lgb_params, cat_cols, seed=42, n_splits=5):
    print(f">> Running LightGBM with Seed: {seed}...")
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)

    oof_preds = np.zeros(len(X_raw))
    test_preds = np.zeros(len(X_test_raw))

    # パラメータにシードをセット
    params = lgb_params.copy()
    params['random_state'] = seed

    for fold, (tr_idx, va_idx) in enumerate(skf.split(X_raw, y)):
        X_tr, y_tr = X_raw.iloc[tr_idx].copy(), y.iloc[tr_idx]
        X_va, y_va = X_raw.iloc[va_idx].copy(), y.iloc[va_idx]
        X_te = X_test_raw.copy()

        # 欠損値処理
        med = X_tr['annual_income'].median()
        X_tr['annual_income'] = X_tr['annual_income'].fillna(med)
        X_va['annual_income'] = X_va['annual_income'].fillna(med)
        X_te['annual_income'] = X_te['annual_income'].fillna(med)

        # 型変換
        X_tr = convert_categorical_types(X_tr, cat_cols)
        X_va = convert_categorical_types(X_va, cat_cols)
        X_te = convert_categorical_types(X_te, cat_cols)

        model = LGBMClassifier(**params)
        model.fit(
            X_tr, y_tr, eval_set=[(X_va, y_va)],
            callbacks=[lgb_early_stopping(stopping_rounds=30, verbose=False)]
        )

        oof_preds[va_idx] = model.predict_proba(X_va)[:, 1]
        test_preds += model.predict_proba(X_te)[:, 1] / n_splits

    cv_score = roc_auc_score(y, oof_preds)
    print(f"   - OOF AUC: {cv_score:.5f}")

    return oof_preds, test_preds

# ==========================================
# 3. Optuna を使ったシード加重平均の最適化
# ==========================================
def optimize_seed_weights(y, oof_dict):
    print("\n--- [3. Optuna シード加重平均の最適化] ---")
    seeds = list(oof_dict.keys())

    def objective(trial):
        # 各シードの重みを探索 (0.0 ~ 1.0)
        weights = [trial.suggest_float(f'w_{s}', 0.0, 1.0) for s in seeds]

        weight_sum = sum(weights)
        if weight_sum == 0:
            raise optuna.exceptions.TrialPruned()

        # 正規化
        weights = [w / weight_sum for w in weights]

        # OOFの予測値を重み付き平均
        blended_oof = np.zeros(len(y))
        for i, s in enumerate(seeds):
            blended_oof += weights[i] * oof_dict[s]

        return roc_auc_score(y, blended_oof)

    optuna.logging.set_verbosity(optuna.logging.WARNING)
    study = optuna.create_study(direction='maximize')
    study.optimize(objective, n_trials=100) # 100回探索

    best_weights_raw = study.best_params
    total = sum(best_weights_raw.values())

    # 正規化した重みを辞書化
    best_weights = {s: best_weights_raw[f'w_{s}'] / total for s in seeds}

    print("✅ 最適な重み:")
    for s, w in best_weights.items():
        print(f"   Seed {s}: {w:.3f}")

    # --- 最終的なOOF予測値を作成してLog Lossを計算 ---
    final_oof = np.zeros(len(y))
    for s in seeds:
        final_oof += best_weights[s] * oof_dict[s]
    final_loss = log_loss(y, final_oof)

    print(f"✅ 最適化後の 最終 OOF AUC: {study.best_value:.5f} / 最終 Log Loss: {final_loss:.5f}")

    return best_weights

# ==========================================
# メイン実行ブロック
# ==========================================
if __name__ == "__main__":
    # データの読み込み
    train_data = pd.read_csv('train.csv')
    test_data = pd.read_csv('test.csv')

    categorical_columns = ['education_level', 'marital_status']

    # ① 安全な特徴量エンジニアリング
    train_df = preprocess_safe_row_wise(train_data)
    test_df = preprocess_safe_row_wise(test_data)

    X_raw = train_df.drop(['customer_id', 'target'], axis=1, errors='ignore')
    y = train_df['target']
    X_test_raw = test_df.drop(['customer_id'], axis=1, errors='ignore')

    # ② LightGBMのハイパーパラメータをOptunaで最適化
    best_lgb_params = tune_lightgbm_with_optuna(X_raw, y, categorical_columns, n_trials=10)

    # ③ 最適化されたパラメータを使って、異なるシードで学習 (OOFとテスト予測を取得)
    print("\n--- [2. 学習フェーズ (LightGBM マルチシード)] ---")
    seeds = [42, 2023, 777]
    oof_results = {}
    test_preds_results = {}

    for s in seeds:
        oof, test_preds = train_lgb_seed(X_raw, y, X_test_raw, best_lgb_params, categorical_columns, seed=s)
        oof_results[s] = oof
        test_preds_results[s] = test_preds

    # ④ Optunaを用いたシード間の加重平均ウェイトの最適化
    best_weights = optimize_seed_weights(y, oof_results)

    # ⑤ テストデータの予測値を最適化された重みでブレンド
    final_predictions = np.zeros(len(X_test_raw))
    for s in seeds:
        final_predictions += best_weights[s] * test_preds_results[s]

    # ⑥ 出力
    submission = pd.DataFrame({'customer_id': test_data['customer_id'], 'target': final_predictions})
    submission.to_csv('final_submission_lgbm_weighted.csv', index=False)
    print("\n✅ 全工程完了: パラメータ最適化 & シード加重平均を施した 'final_submission_lgbm_weighted.csv' を出力しました！")


--- [1. Optuna パラメータ最適化 (LightGBM)] ---
✅ 最適パラメータ: {'learning_rate': 0.014738153189731524, 'max_depth': 7, 'num_leaves': 55, 'min_child_samples': 29, 'n_estimators': 1000, 'class_weight': 'balanced', 'verbose': -1}

--- [2. 学習フェーズ (LightGBM マルチシード)] ---
>> Running LightGBM with Seed: 42...
   - OOF AUC: 0.87406
>> Running LightGBM with Seed: 2023...
   - OOF AUC: 0.86963
>> Running LightGBM with Seed: 777...
   - OOF AUC: 0.87415

--- [3. Optuna シード加重平均の最適化] ---
✅ 最適な重み:
   Seed 42: 0.375
   Seed 2023: 0.281
   Seed 777: 0.344
✅ 最適化後の 最終 OOF AUC: 0.88318 / 最終 Log Loss: 0.30487

✅ 全工程完了: パラメータ最適化 & シード加重平均を施した 'final_submission_lgbm_weighted.csv' を出力しました！
